
# Projet Machine Learning complet et expliqué

## Classification de tumeurs mammaires

Ce notebook présente toutes les étapes principales d'un projet de **Machine Learning supervisé**.

### Objectif

À partir de mesures calculées sur une tumeur, nous voulons prédire si elle est :

- **bénigne** ;
- **maligne**.

Il s'agit donc d'un problème de **classification binaire**.

### Étapes du projet

1. Comprendre le problème métier.
2. Charger les données.
3. Explorer le dataset.
4. Vérifier la qualité des données.
5. Séparer les variables explicatives et la cible.
6. Créer un jeu d'entraînement et un jeu de test.
7. Construire une première référence avec un modèle naïf.
8. Préparer les données.
9. Entraîner un modèle de classification.
10. Évaluer ses performances.
11. Effectuer une validation croisée.
12. Interpréter le modèle.
13. Tester une prédiction.
14. Sauvegarder le pipeline entraîné.

> Ce notebook est conçu comme un support pédagogique.  
> Les commentaires expliquent non seulement **ce que fait le code**, mais aussi **pourquoi nous le faisons**.



## 1. Installation éventuelle des bibliothèques

Dans un environnement Jupyter neuf, les bibliothèques suivantes peuvent être nécessaires :

```bash
pip install pandas numpy matplotlib scikit-learn joblib
```

La cellule suivante importe les outils utilisés dans le projet.


In [ ]:

# Manipulation des données
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt

# Dataset d'exemple fourni directement par scikit-learn
from sklearn.datasets import load_breast_cancer

# Séparation des données
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold

# Prétraitement et pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Modèles
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

# Métriques d'évaluation
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    RocCurveDisplay
)

# Sauvegarde du modèle
import joblib

# Réglage de l'affichage pandas
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

print("Bibliothèques importées avec succès.")



## 2. Chargement du dataset

Nous utilisons le dataset public **Breast Cancer Wisconsin**, intégré à `scikit-learn`.

Chaque ligne représente une observation.  
Chaque colonne décrit une caractéristique mesurée sur une image de cellule.

La variable cible contient deux classes :

- `0` : tumeur maligne ;
- `1` : tumeur bénigne.

L'intérêt d'utiliser un dataset intégré à scikit-learn est qu'il est immédiatement disponible, documenté et reproductible.


In [ ]:

# Chargement du dataset sous forme d'objet scikit-learn
dataset = load_breast_cancer()

# Création d'un DataFrame avec les variables explicatives
df = pd.DataFrame(dataset.data, columns=dataset.feature_names)

# Ajout de la variable cible
df["target"] = dataset.target

# Ajout d'un libellé lisible pour faciliter l'exploration
df["diagnostic"] = df["target"].map({
    0: "maligne",
    1: "bénigne"
})

print(f"Nombre de lignes : {df.shape[0]}")
print(f"Nombre de colonnes : {df.shape[1]}")
df.head()



## 3. Compréhension de la structure des données

Avant de créer un modèle, il faut comprendre le contenu du dataset.

Nous allons vérifier :

- les dimensions ;
- les noms des colonnes ;
- les types de données ;
- la présence de valeurs manquantes ;
- les statistiques descriptives ;
- la répartition de la cible.


In [ ]:

# Affichage des informations générales :
# - nombre de lignes non nulles ;
# - type de chaque colonne ;
# - mémoire utilisée.
df.info()


In [ ]:

# Statistiques descriptives des variables numériques
# count : nombre de valeurs
# mean  : moyenne
# std   : écart-type
# min   : minimum
# 25%, 50%, 75% : quartiles
# max   : maximum
df.describe().T.head(10)


In [ ]:

# Comptage des valeurs manquantes par colonne
missing_values = df.isna().sum().sort_values(ascending=False)

print("Nombre total de valeurs manquantes :", missing_values.sum())
missing_values.head(10)



### Pourquoi vérifier les valeurs manquantes ?

De nombreux algorithmes ne savent pas traiter directement les cellules vides.

Selon le contexte, nous pourrions :

- supprimer certaines lignes ;
- supprimer une colonne peu utile ;
- remplacer les valeurs manquantes par une moyenne ou une médiane ;
- utiliser une méthode d'imputation plus avancée.

Dans ce dataset, aucune valeur n'est manquante.


In [ ]:

# Répartition de la variable cible
class_counts = df["diagnostic"].value_counts()
class_percentages = df["diagnostic"].value_counts(normalize=True).mul(100)

distribution = pd.DataFrame({
    "nombre": class_counts,
    "pourcentage": class_percentages
})

distribution


In [ ]:

# Représentation graphique de la répartition des classes
distribution["nombre"].plot(kind="bar")

plt.title("Répartition des classes")
plt.xlabel("Diagnostic")
plt.ylabel("Nombre d'observations")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()



### Interprétation de la répartition des classes

Les classes ne sont pas parfaitement équilibrées, mais les deux catégories sont suffisamment représentées.

Cette vérification est importante car une forte majorité d'une seule classe peut rendre l'**accuracy** trompeuse.

Exemple : avec 95 % de cas bénins, un modèle prédisant toujours « bénigne » obtiendrait 95 % d'accuracy sans avoir appris à reconnaître les cas malins.



## 4. Exploration de quelques variables

Nous allons comparer certaines mesures selon le diagnostic.

L'objectif n'est pas encore d'entraîner un modèle, mais de voir si certaines variables semblent différencier les deux classes.


In [ ]:

# Sélection de quelques colonnes faciles à visualiser
features_to_plot = [
    "mean radius",
    "mean texture",
    "mean perimeter",
    "mean area"
]

# Création d'un histogramme distinct pour chaque variable
for feature in features_to_plot:
    df[df["diagnostic"] == "bénigne"][feature].plot(
        kind="hist",
        bins=25,
        alpha=0.6,
        label="bénigne"
    )
    df[df["diagnostic"] == "maligne"][feature].plot(
        kind="hist",
        bins=25,
        alpha=0.6,
        label="maligne"
    )

    plt.title(f"Distribution de {feature}")
    plt.xlabel(feature)
    plt.ylabel("Fréquence")
    plt.legend()
    plt.tight_layout()
    plt.show()



## 5. Définition de `X` et `y`

Dans un projet supervisé :

- `X` contient les **variables explicatives**, aussi appelées caractéristiques ou features ;
- `y` contient la **variable cible**, c'est-à-dire ce que le modèle doit prédire.

Nous retirons les colonnes `target` et `diagnostic` de `X` car elles donnent directement la réponse.

Conserver la cible dans les variables explicatives provoquerait une **fuite de données** (*data leakage*). Le modèle aurait accès à la réponse pendant l'entraînement.


In [ ]:

# Variables explicatives
X = df.drop(columns=["target", "diagnostic"])

# Variable cible
y = df["target"]

print("Dimensions de X :", X.shape)
print("Dimensions de y :", y.shape)
print("Classes présentes :", sorted(y.unique()))



## 6. Séparation entraînement / test

Nous séparons le dataset en deux parties :

- **jeu d'entraînement** : utilisé pour apprendre les paramètres du modèle ;
- **jeu de test** : utilisé uniquement à la fin pour évaluer le modèle sur des données qu'il n'a jamais vues.

Nous utilisons :

- `test_size=0.20` : 20 % des données pour le test ;
- `random_state=42` : rend la séparation reproductible ;
- `stratify=y` : conserve une proportion similaire de chaque classe dans les deux jeux.


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Taille du jeu d'entraînement :", X_train.shape)
print("Taille du jeu de test :", X_test.shape)

print("\nRépartition dans le jeu d'entraînement :")
print(y_train.value_counts(normalize=True).sort_index())

print("\nRépartition dans le jeu de test :")
print(y_test.value_counts(normalize=True).sort_index())



## 7. Création d'une baseline

Une **baseline** est une référence simple.

Avant de juger un modèle sophistiqué, il faut vérifier qu'il fait mieux qu'une stratégie naïve.

Le `DummyClassifier` utilisé ici prédit toujours la classe majoritaire.


In [ ]:

# Modèle naïf : prédit toujours la classe la plus fréquente
dummy_model = DummyClassifier(strategy="most_frequent")

# Entraînement
dummy_model.fit(X_train, y_train)

# Prédictions
dummy_predictions = dummy_model.predict(X_test)

# Évaluation
dummy_accuracy = accuracy_score(y_test, dummy_predictions)
dummy_balanced_accuracy = balanced_accuracy_score(y_test, dummy_predictions)
dummy_f1 = f1_score(y_test, dummy_predictions)

print(f"Accuracy de la baseline          : {dummy_accuracy:.4f}")
print(f"Balanced accuracy de la baseline : {dummy_balanced_accuracy:.4f}")
print(f"F1-score de la baseline          : {dummy_f1:.4f}")



## 8. Prétraitement et pipeline

Nous allons entraîner une **régression logistique**.

Malgré son nom, la régression logistique est un algorithme de classification.

Elle calcule une probabilité d'appartenance à une classe.

### Pourquoi standardiser les données ?

Les variables n'ont pas toutes la même échelle :

- certaines valeurs peuvent être proches de `0.01` ;
- d'autres peuvent dépasser `1000`.

La standardisation transforme chaque variable pour qu'elle ait approximativement :

- une moyenne égale à 0 ;
- un écart-type égal à 1.

### Pourquoi utiliser un pipeline ?

Le pipeline regroupe :

1. la standardisation ;
2. le modèle.

Cela garantit que les mêmes transformations sont appliquées pendant :

- l'entraînement ;
- la validation croisée ;
- les prédictions futures.

Le pipeline réduit aussi les risques de fuite de données.


In [ ]:

# Création du pipeline complet
model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        random_state=42
    ))
])

# Entraînement du pipeline sur le jeu d'entraînement
model.fit(X_train, y_train)

print("Pipeline entraîné avec succès.")



## 9. Prédictions sur le jeu de test

Le modèle produit deux types de sorties :

- `predict()` : classe finale prédite ;
- `predict_proba()` : probabilité estimée pour chaque classe.

Pour le calcul de la courbe ROC et du ROC-AUC, nous utilisons la probabilité de la classe positive.


In [ ]:

# Classe prédite pour chaque observation du jeu de test
y_pred = model.predict(X_test)

# Probabilité estimée pour la classe 1
y_proba = model.predict_proba(X_test)[:, 1]

print("Exemples de classes prédites :", y_pred[:10])
print("Exemples de probabilités     :", np.round(y_proba[:10], 4))



## 10. Calcul des métriques

### Accuracy

Proportion totale de prédictions correctes.

### Balanced accuracy

Moyenne du rappel obtenu dans chaque classe.  
Elle est utile lorsque les classes sont déséquilibrées.

### Precision

Parmi les observations prédites positives, combien sont réellement positives ?

### Recall

Parmi les observations réellement positives, combien ont été détectées ?

Dans un contexte médical, le choix de la classe positive doit être clairement défini.  
Dans notre dataset, la classe `1` correspond à « bénigne ». Nous afficherons donc aussi le rapport par classe pour éviter toute ambiguïté.

### F1-score

Moyenne harmonique entre précision et rappel.

### ROC-AUC

Mesure la capacité du modèle à classer les observations positives au-dessus des négatives, indépendamment d'un seuil unique.


In [ ]:

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
    "precision_classe_1": precision_score(y_test, y_pred),
    "recall_classe_1": recall_score(y_test, y_pred),
    "f1_classe_1": f1_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_proba)
}

metrics_df = pd.DataFrame(
    metrics.items(),
    columns=["métrique", "score"]
)

metrics_df


In [ ]:

# Rapport détaillé par classe
# target_names suit l'ordre des classes : 0 puis 1
print(classification_report(
    y_test,
    y_pred,
    target_names=["maligne", "bénigne"],
    digits=4
))



## 11. Matrice de confusion

La matrice de confusion compare les vraies classes et les classes prédites.

Dans notre cas :

- ligne « maligne » : vrais cas malins ;
- ligne « bénigne » : vrais cas bénins ;
- colonnes : prédictions du modèle.

Elle permet de voir précisément le type d'erreur commis.


In [ ]:

cm = confusion_matrix(y_test, y_pred)

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["maligne", "bénigne"]
)

display.plot(values_format="d")
plt.title("Matrice de confusion")
plt.tight_layout()
plt.show()



## 12. Courbe ROC

La courbe ROC représente le compromis entre :

- le taux de vrais positifs ;
- le taux de faux positifs.

Plus la courbe se rapproche du coin supérieur gauche, meilleur est le modèle.

Un ROC-AUC proche de :

- `0.5` correspond à un classement aléatoire ;
- `1.0` correspond à un classement parfait.


In [ ]:

RocCurveDisplay.from_predictions(
    y_test,
    y_proba,
    name="Régression logistique"
)

plt.title("Courbe ROC")
plt.tight_layout()
plt.show()



## 13. Validation croisée stratifiée

Une seule séparation entraînement/test peut donner un résultat dépendant du hasard.

La **validation croisée** découpe le jeu d'entraînement en plusieurs plis.

Avec une validation croisée à 5 plis :

1. quatre plis servent à l'entraînement ;
2. le cinquième sert à la validation ;
3. l'opération est répétée cinq fois ;
4. chaque pli sert une fois de validation.

La version `StratifiedKFold` conserve la proportion des classes dans chaque pli.

Nous calculons plusieurs métriques afin d'avoir une vision plus robuste des performances.


In [ ]:

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_results = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False
)

cv_summary = pd.DataFrame({
    metric.replace("test_", ""): [
        np.mean(values),
        np.std(values)
    ]
    for metric, values in cv_results.items()
    if metric.startswith("test_")
}, index=["moyenne", "écart-type"]).T

cv_summary



### Comment lire la validation croisée ?

- Une moyenne élevée indique de bonnes performances générales.
- Un faible écart-type indique que les résultats sont stables d'un pli à l'autre.
- Un score élevé avec un très grand écart-type doit être interprété avec prudence.

La validation croisée est utilisée pendant la phase de conception.  
Le jeu de test reste séparé pour l'évaluation finale.



## 14. Interprétation des variables

La régression logistique apprend un coefficient pour chaque variable.

- coefficient positif : pousse la prédiction vers la classe `1`, ici « bénigne » ;
- coefficient négatif : pousse la prédiction vers la classe `0`, ici « maligne » ;
- valeur absolue élevée : influence plus importante dans le modèle.

Comme les variables ont été standardisées, les coefficients sont plus facilement comparables.


In [ ]:

# Récupération du modèle entraîné à l'intérieur du pipeline
logistic_regression = model.named_steps["classifier"]

# Association des coefficients aux noms des variables
coefficients = pd.DataFrame({
    "variable": X.columns,
    "coefficient": logistic_regression.coef_[0]
})

# Ajout de la valeur absolue pour classer les variables par importance
coefficients["importance_absolue"] = coefficients["coefficient"].abs()

top_coefficients = coefficients.sort_values(
    by="importance_absolue",
    ascending=False
).head(15)

top_coefficients


In [ ]:

# Visualisation des coefficients les plus influents
plot_data = top_coefficients.sort_values("coefficient")

plot_data.plot(
    x="variable",
    y="coefficient",
    kind="barh",
    legend=False
)

plt.title("Variables les plus influentes")
plt.xlabel("Coefficient de la régression logistique")
plt.ylabel("Variable")
plt.tight_layout()
plt.show()



## 15. Prédiction d'une nouvelle observation

Pour simuler l'utilisation du modèle, nous prenons une ligne du jeu de test.

Dans un vrai projet, cette ligne pourrait provenir :

- d'un formulaire ;
- d'une API ;
- d'un fichier CSV ;
- d'une base de données ;
- d'une application métier.

Grâce au pipeline, nous n'avons pas besoin d'appliquer manuellement le `StandardScaler`.


In [ ]:

# Sélection d'une observation sous forme de DataFrame
new_observation = X_test.iloc[[0]]

predicted_class = model.predict(new_observation)[0]
predicted_probabilities = model.predict_proba(new_observation)[0]

label = {
    0: "maligne",
    1: "bénigne"
}[predicted_class]

print("Classe prédite :", label)
print(f"Probabilité classe maligne : {predicted_probabilities[0]:.4f}")
print(f"Probabilité classe bénigne : {predicted_probabilities[1]:.4f}")
print("Classe réelle :", "bénigne" if y_test.iloc[0] == 1 else "maligne")



## 16. Sauvegarde du pipeline entraîné

Nous sauvegardons l'ensemble du pipeline :

- standardisation ;
- modèle ;
- paramètres appris.

Le fichier obtenu peut ensuite être rechargé dans une application Python, une API FastAPI ou une interface Streamlit.


In [ ]:

model_path = "modele_classification_tumeurs.joblib"

# Sauvegarde
joblib.dump(model, model_path)

print(f"Pipeline sauvegardé dans : {model_path}")


In [ ]:

# Exemple de rechargement
loaded_model = joblib.load(model_path)

# Vérification rapide : le modèle rechargé doit produire la même prédiction
loaded_prediction = loaded_model.predict(new_observation)[0]

print("Prédiction du modèle rechargé :", loaded_prediction)
print("Prédiction identique :", loaded_prediction == predicted_class)



## 17. Bilan du projet

Nous avons réalisé un pipeline complet de Machine Learning :

1. définition du problème ;
2. chargement d'un dataset public ;
3. exploration des données ;
4. vérification de leur qualité ;
5. création de `X` et `y` ;
6. séparation entraînement/test ;
7. construction d'une baseline ;
8. standardisation des variables ;
9. entraînement d'une régression logistique ;
10. calcul des métriques ;
11. matrice de confusion ;
12. courbe ROC ;
13. validation croisée ;
14. interprétation des coefficients ;
15. prédiction d'une nouvelle observation ;
16. sauvegarde du pipeline.

## Points essentiels à retenir

### Le modèle n'est qu'une partie du projet

Un projet ML comprend également :

- la compréhension métier ;
- la qualité des données ;
- le choix des métriques ;
- la validation ;
- l'interprétation ;
- le déploiement ;
- la surveillance future du modèle.

### Le jeu de test ne doit pas servir à entraîner

Il représente des données inconnues utilisées pour l'évaluation finale.

### Le pipeline sécurise les transformations

Il applique exactement les mêmes traitements aux données d'entraînement et aux futures données.

### Une seule métrique ne suffit pas

Il faut souvent combiner :

- accuracy ;
- balanced accuracy ;
- précision ;
- rappel ;
- F1-score ;
- matrice de confusion ;
- ROC-AUC.

### Les métriques dépendent du contexte métier

Dans un contexte médical, certaines erreurs sont beaucoup plus graves que d'autres.  
Il faut donc identifier la classe critique et choisir les métriques en conséquence.



## 18. Exercices proposés

Pour approfondir, tu peux modifier le notebook afin de :

1. remplacer la régression logistique par un `RandomForestClassifier` ;
2. comparer les performances des deux modèles ;
3. rechercher les meilleurs hyperparamètres avec `GridSearchCV` ;
4. afficher les faux positifs et les faux négatifs ;
5. modifier le seuil de classification ;
6. créer une fonction `predict_tumor()` ;
7. exposer le modèle avec FastAPI ;
8. créer une interface Streamlit ;
9. enregistrer les prédictions dans PostgreSQL ;
10. ajouter un suivi des expériences avec MLflow.

Ces extensions correspondent aux étapes que l'on retrouve dans des projets ML professionnels.
